In [4]:
import torch
import subprocess

print("=== GPU INFO ===")
print(torch.cuda.get_device_name(0))
print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n=== VERSIONS ===")
print(f"PyTorch: {torch.__version__}")
print(subprocess.getoutput("python --version"))
print(subprocess.getoutput("pip show ultralytics | grep Version"))
print(subprocess.getoutput("pip show tensorrt | grep Version"))

print("\n=== FILES ===")
import os
for f in os.listdir("/home/jj7/Scalable"):
    size = os.path.getsize(f"/home/jj7/Scalable/{f}") / 1e6
    print(f"  {f}  ({size:.1f} MB)")

=== GPU INFO ===
NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb
Compute Capability: (12, 0)
VRAM: 25.4 GB

=== VERSIONS ===
PyTorch: 2.10.0+cu128
Python 3.13.9
Version: 8.4.19


=== FILES ===
  .ipynb_checkpoints  (0.0 MB)
  yolo11s_pruned_1.pt.zip  (10.5 MB)
  YOLO26_pruned_20_final.pt  (9.4 MB)
  Quantization.ipynb  (0.0 MB)


In [5]:
import subprocess, sys, zipfile, os

print("=== Installing TensorRT ===")
result = subprocess.run([
    sys.executable, "-m", "pip", "install",
    "tensorrt", "tensorrt-cu12", "tensorrt-cu12-bindings", "tensorrt-cu12-libs",
    "--extra-index-url", "https://pypi.nvidia.com",
    "-q"
], capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else "")
print(result.stderr[-300:] if result.stderr else "")

print("\n=== Installing onnx + onnxruntime-gpu ===")
result2 = subprocess.run([
    sys.executable, "-m", "pip", "install",
    "onnx", "onnxruntime-gpu", "-q"
], capture_output=True, text=True)
print(result2.stdout[-300:] if result2.stdout else "Done")

print("\n=== Unzipping YOLO11 ===")
zip_path = "/home/jj7/Scalable/yolo11s_pruned_1.pt.zip"
extract_dir = "/home/jj7/Scalable/yolo11s_extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Extracted contents:")
for root, dirs, files in os.walk(extract_dir):
    for f in files:
        fpath = os.path.join(root, f)
        print(f"  {fpath}  ({os.path.getsize(fpath)/1e6:.2f} MB)")

print("\n=== Verifying TensorRT install ===")
result3 = subprocess.run([sys.executable, "-m", "pip", "show", "tensorrt"],
    capture_output=True, text=True)
print(result3.stdout)

=== Installing TensorRT ===



=== Installing onnx + onnxruntime-gpu ===
Done

=== Unzipping YOLO11 ===
Extracted contents:
  /home/jj7/Scalable/yolo11s_extracted/best/data.pkl  (0.14 MB)
  /home/jj7/Scalable/yolo11s_extracted/best/.format_version  (0.00 MB)
  /home/jj7/Scalable/yolo11s_extracted/best/.storage_alignment  (0.00 MB)
  /home/jj7/Scalable/yolo11s_extracted/best/byteorder  (0.00 MB)
  /home/jj7/Scalable/yolo11s_extracted/best/version  (0.00 MB)
  /home/jj7/Scalable/yolo11s_extracted/best/data/0  (0.00 MB)
  /home/jj7/Scalable/yolo11s_extracted/best/data/1  (0.00 MB)
  /home/jj7/Scalable/yolo11s_extracted/best/data/2  (0.00 MB)
  /home/jj7/Scalable/yolo11s_extracted/best/data/3  (0.00 MB)
  /home/jj7/Scalable/yolo11s_extracted/best/data/4  (0.00 MB)
  /home/jj7/Scalable/yolo11s_extracted/best/data/5  (0.00 MB)
  /home/jj7/Scalable/yolo11s_extracted/best/data/6  (0.03 MB)
  /home/jj7/Scalable/yolo11s_extracted/best/data/7  (0.00 MB)
  /home/jj7/Scalable/yolo11s_extracted/best

In [8]:
import torch
import shutil
from ultralytics import YOLO

# PyTorch zip-format checkpoints must be loaded directly from the ZIP
# — NOT from the extracted folder. Load the zip file itself directly.

print("=== Loading YOLO11s from zip directly ===")
ckpt11 = torch.load(
    "/home/jj7/Scalable/yolo11s_pruned_1.pt.zip",
    weights_only=False,
    map_location="cpu"
)
print(f"Checkpoint keys: {list(ckpt11.keys()) if isinstance(ckpt11, dict) else type(ckpt11)}")

# Save as a proper flat .pt file
save_path11 = "/home/jj7/Scalable/yolo11s_pruned.pt"
torch.save(ckpt11, save_path11)
print(f"Saved → {save_path11}  ({os.path.getsize(save_path11)/1e6:.1f} MB)")

# Now load with YOLO
model11 = YOLO(save_path11)
print("YOLO11 loaded ✓")
print(f"  Task : {model11.task}")
print(f"  Names: {model11.names}")

# YOLO26 — direct
print("\n=== Loading YOLO26 ===")
model26 = YOLO("/home/jj7/Scalable/YOLO26_pruned_20_final.pt")
print("YOLO26 loaded ✓")
print(f"  Task : {model26.task}")
print(f"  Names: {model26.names}")

print("\n=== Param counts ===")
print(f"YOLO11: {sum(p.numel() for p in model11.model.parameters()):,}")
print(f"YOLO26: {sum(p.numel() for p in model26.model.parameters()):,}")

=== Loading YOLO11s from zip directly ===
Checkpoint keys: ['date', 'version', 'license', 'docs', 'epoch', 'best_fitness', 'model', 'ema', 'updates', 'optimizer', 'scaler', 'train_args', 'train_metrics', 'train_results', 'git']
Saved → /home/jj7/Scalable/yolo11s_pruned.pt  (16.6 MB)
YOLO11 loaded ✓
  Task : detect
  Names: {0: 'pedestrian', 1: 'cyclist', 2: 'car', 3: 'large_vehicle'}

=== Loading YOLO26 ===
YOLO26 loaded ✓
  Task : detect
  Names: {0: 'pedestrian', 1: 'cyclist', 2: 'car', 3: 'large_vehicle'}

=== Param counts ===
YOLO11: 8,138,542
YOLO26: 8,514,378


In [9]:
import os
from ultralytics import YOLO

# Reload models cleanly
model11 = YOLO("/home/jj7/Scalable/yolo11s_pruned.pt")
model26 = YOLO("/home/jj7/Scalable/YOLO26_pruned_20_final.pt")

print("=" * 50)
print("Exporting YOLO11s → ONNX (FP16)")
print("=" * 50)
path11 = model11.export(
    format="onnx",
    imgsz=640,
    half=True,        # FP16
    dynamic=False,
    simplify=True,
    device=0,
    batch=1,
)
print(f"YOLO11 ONNX saved → {path11}")
print(f"Size: {os.path.getsize(path11)/1e6:.1f} MB")

print()
print("=" * 50)
print("Exporting YOLO26 → ONNX (FP16)")
print("=" * 50)
path26 = model26.export(
    format="onnx",
    imgsz=640,
    half=True,
    dynamic=False,
    simplify=True,
    device=0,
    batch=1,
)
print(f"YOLO26 ONNX saved → {path26}")
print(f"Size: {os.path.getsize(path26)/1e6:.1f} MB")

print("\n✅ Both ONNX exports done!")

Exporting YOLO11s → ONNX (FP16)
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO11s_pruned summary (fused): 101 layers, 8,124,647 parameters, 0 gradients, 16.3 GFLOPs

PyTorch: starting from '/home/jj7/Scalable/yolo11s_pruned.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (15.8 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.71'] not found, attempting AutoUpdate...
Defaulting to user installation because normal site-packages is not writeable

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxslim]


requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.90...


2026-03-29 13:11:57.092200592 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2822399, index: 0, mask: {1, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-29 13:11:57.096160443 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2822400, index: 1, mask: {2, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-29 13:11:57.100142585 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2822401, index: 2, mask: {3, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-29 13:11:57.104140819 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2822402, index: 3, mask: {4, }, error code: 22 error msg: Invalid argument. Specify the n

ONNX: export success ✅ 2.9s, saved as '/home/jj7/Scalable/yolo11s_pruned.onnx' (15.7 MB)

Export complete (3.5s)
Results saved to /home/jj7/Scalable
Predict:         yolo predict task=detect model=/home/jj7/Scalable/yolo11s_pruned.onnx imgsz=640 half
Validate:        yolo val task=detect model=/home/jj7/Scalable/yolo11s_pruned.onnx imgsz=640 data=pedestrian.yaml half 
Visualize:       https://netron.app
YOLO11 ONNX saved → /home/jj7/Scalable/yolo11s_pruned.onnx
Size: 16.4 MB

Exporting YOLO26 → ONNX (FP16)
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO26s_pruned_20pct summary (fused): 120 layers, 8,498,697 parameters, 0 gradients, 16.4 GFLOPs

PyTorch: starting from '/home/jj7/Scalable/YOLO26_pruned_20_final.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (16.6 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export succ

2026-03-29 13:11:58.245147518 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2822497, index: 0, mask: {1, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-29 13:11:58.245187683 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2822498, index: 1, mask: {2, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-29 13:11:58.245195283 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2822499, index: 2, mask: {3, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-29 13:11:58.245202583 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2822500, index: 3, mask: {4, }, error code: 22 error msg: Invalid argument. Specify the n

In [11]:
import tensorrt as trt
import os

def build_fp16_engine(onnx_path, engine_path, workspace_gb=4):
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, logger)

    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(f"  Parse error: {parser.get_error(i)}")
            raise RuntimeError("ONNX parse failed")

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, workspace_gb * (1 << 30))
    config.set_flag(trt.BuilderFlag.FP16)   # FP16 only — no calibrator needed

    print(f"  Building FP16 engine from {os.path.basename(onnx_path)} ...")
    serialized = builder.build_serialized_network(network, config)
    if serialized is None:
        raise RuntimeError("Engine build failed")

    with open(engine_path, "wb") as f:
        f.write(serialized)
    print(f"  Saved → {engine_path}  ({os.path.getsize(engine_path)/1e6:.1f} MB)")
    return engine_path


# INT8 with random-image calibrator
import numpy as np

class RandomCalibrator(trt.IInt8EntropyCalibrator2):
    def __init__(self, n_batches=50, batch_size=1, imgsz=640):
        super().__init__()
        self.n_batches = n_batches
        self.batch_size = batch_size
        self.imgsz = imgsz
        self.current = 0
        self.cache_file = "/tmp/calib_cache.bin"
        import ctypes
        nbytes = batch_size * 3 * imgsz * imgsz * 2  # FP16
        self.device_input = trt.volume((batch_size, 3, imgsz, imgsz))
        import pycuda.driver as cuda
        import pycuda.autoinit
        self._cuda = cuda
        self.d_input = cuda.mem_alloc(batch_size * 3 * imgsz * imgsz * np.dtype(np.float16).itemsize)

    def get_batch_size(self): return self.batch_size

    def get_batch(self, names):
        if self.current >= self.n_batches:
            return None
        batch = (np.random.rand(self.batch_size, 3, self.imgsz, self.imgsz)
                 .astype(np.float16))
        self._cuda.memcpy_htod(self.d_input, batch)
        self.current += 1
        return [int(self.d_input)]

    def read_calibration_cache(self):
        if os.path.exists(self.cache_file):
            with open(self.cache_file, "rb") as f:
                return f.read()

    def write_calibration_cache(self, cache):
        with open(self.cache_file, "wb") as f:
            f.write(cache)


def build_int8_engine_calibrated(onnx_path, engine_path, workspace_gb=4):
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser = trt.OnnxParser(network, logger)
    with open(onnx_path, "rb") as f:
        parser.parse(f.read())

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, workspace_gb * (1 << 30))
    config.set_flag(trt.BuilderFlag.INT8)
    config.set_flag(trt.BuilderFlag.FP16)

    try:
        calib = RandomCalibrator(n_batches=50)
        config.int8_calibrator = calib
        print(f"  Building INT8 engine with calibrator ...")
    except Exception as e:
        print(f"  pycuda not available ({e}), falling back to FP16 only")
        config.clear_flag(trt.BuilderFlag.INT8)

    serialized = builder.build_serialized_network(network, config)
    if serialized is None:
        raise RuntimeError("Engine build failed")
    with open(engine_path, "wb") as f:
        f.write(serialized)
    print(f"  Saved → {engine_path}  ({os.path.getsize(engine_path)/1e6:.1f} MB)")
    return engine_path


# ── Run FP16 first (guaranteed to work) ──────────────────────────
print("="*55)
print("YOLO11s → FP16 engine")
print("="*55)
e11_fp16 = build_fp16_engine(
    "/home/jj7/Scalable/yolo11s_pruned.onnx",
    "/home/jj7/Scalable/yolo11s_fp16.engine"
)

print("\n" + "="*55)
print("YOLO26 → FP16 engine")
print("="*55)
e26_fp16 = build_fp16_engine(
    "/home/jj7/Scalable/YOLO26_pruned_20_final.onnx",
    "/home/jj7/Scalable/YOLO26_fp16.engine"
)

print("\n✅ FP16 engines done!")
print(f"  YOLO11 → {e11_fp16}")
print(f"  YOLO26 → {e26_fp16}")

# ── Now try INT8 with calibrator ─────────────────────────────────
print("\n" + "="*55)
print("YOLO11s → INT8 engine (with calibration)")
print("="*55)
try:
    e11_int8 = build_int8_engine_calibrated(
        "/home/jj7/Scalable/yolo11s_pruned.onnx",
        "/home/jj7/Scalable/yolo11s_int8.engine"
    )
    e26_int8 = build_int8_engine_calibrated(
        "/home/jj7/Scalable/YOLO26_pruned_20_final.onnx",
        "/home/jj7/Scalable/YOLO26_int8.engine"
    )
    print("✅ INT8 engines done!")
except Exception as ex:
    print(f"⚠️  INT8 failed: {ex}")
    print("   FP16 engines are ready and still give 1.5-2x speedup on Blackwell.")

YOLO11s → FP16 engine
[03/29/2026-13:13:47] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
  Building FP16 engine from yolo11s_pruned.onnx ...
  Saved → /home/jj7/Scalable/yolo11s_fp16.engine  (19.7 MB)

YOLO26 → FP16 engine
[03/29/2026-13:14:01] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
  Building FP16 engine from YOLO26_pruned_20_final.onnx ...
  Saved → /home/jj7/Scalable/YOLO26_fp16.engine  (20.5 MB)

✅ FP16 engines done!
  YOLO11 → /home/jj7/Scalable/yolo11s_fp16.engine
  YOLO26 → /home/jj7/Scalable/YOLO26_fp16.engine

YOLO11s → INT8 

In [13]:
import os
from ultralytics import YOLO

# We need a small dataset of real images for calibration
# Let's download COCO128 (128 images, ~6MB) as calibration data
import subprocess
print("=== Downloading calibration images (COCO128) ===")
r = subprocess.run([
    "python", "-c",
    """
from ultralytics.utils.downloads import download
download('https://ultralytics.com/assets/coco128.zip', dir='/tmp/')
import zipfile, os
with zipfile.ZipFile('/tmp/coco128.zip', 'r') as z:
    z.extractall('/tmp/')
print('Images:', len(os.listdir('/tmp/coco128/images/train2017')))
"""
], capture_output=True, text=True, timeout=120)
print(r.stdout)
print(r.stderr[-300:] if r.stderr else "")

=== Downloading calibration images (COCO128) ===


Unzipping /tmp/coco128.zip to /tmp/coco128...: 100% ━━━━━━━━━━━━ 263/263 12.7Kfiles/s 0.0s
Images: 128




In [14]:
from ultralytics import YOLO
import os

# Create a minimal data.yaml for calibration
data_yaml = """\
path: /tmp/coco128
train: images/train2017
val: images/train2017
nc: 4
names: ['pedestrian', 'cyclist', 'car', 'large_vehicle']
"""
with open("/tmp/calib_data.yaml", "w") as f:
    f.write(data_yaml)
print("data.yaml written ✅")

# ── YOLO11 INT8 export ────────────────────────────────────────────
print("\n" + "="*55)
print("YOLO11s → TensorRT INT8 (Ultralytics export)")
print("="*55)
model11 = YOLO("/home/jj7/Scalable/yolo11s_pruned.pt")
path11 = model11.export(
    format="engine",
    imgsz=640,
    int8=True,
    data="/tmp/calib_data.yaml",
    device=0,
    batch=1,
    workspace=4,
    verbose=False
)
print(f"YOLO11 INT8 engine → {path11}")

# ── YOLO26 INT8 export ────────────────────────────────────────────
print("\n" + "="*55)
print("YOLO26 → TensorRT INT8 (Ultralytics export)")
print("="*55)
model26 = YOLO("/home/jj7/Scalable/YOLO26_pruned_20_final.pt")
path26 = model26.export(
    format="engine",
    imgsz=640,
    int8=True,
    data="/tmp/calib_data.yaml",
    device=0,
    batch=1,
    workspace=4,
    verbose=False
)
print(f"YOLO26 INT8 engine → {path26}")

print("\n✅ Done! Files in /home/jj7/Scalable/:")
for f in os.listdir("/home/jj7/Scalable"):
    if f.endswith(".engine") or f.endswith(".onnx"):
        size = os.path.getsize(f"/home/jj7/Scalable/{f}") / 1e6
        print(f"  {f:45s} {size:.1f} MB")

data.yaml written ✅

YOLO11s → TensorRT INT8 (Ultralytics export)
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO11s_pruned summary (fused): 101 layers, 8,124,647 parameters, 0 gradients, 16.3 GFLOPs

PyTorch: starting from '/home/jj7/Scalable/yolo11s_pruned.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (15.8 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.90...


2026-03-29 13:19:12.432152038 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2827907, index: 0, mask: {1, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-29 13:19:12.436133449 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2827908, index: 1, mask: {2, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-29 13:19:12.437252682 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2827909, index: 2, mask: {3, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-29 13:19:12.441133335 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2827910, index: 3, mask: {4, }, error code: 22 error msg: Invalid argument. Specify the n

ONNX: export success ✅ 0.8s, saved as '/home/jj7/Scalable/yolo11s_pruned.onnx' (31.3 MB)

TensorRT: starting export with TensorRT 10.16.0.72...
TensorRT: collecting INT8 calibration images from 'data=/tmp/calib_data.yaml'
Fast image access ✅ (ping: 0.0±0.0 ms, read: 3654.1±1688.2 MB/s, size: 56.1 KB)
Scanning /tmp/coco128/labels/train2017... 126 images, 2 backgrounds, 123 corrupt: 100% ━━━━━━━━━━━━ 128/128 16.9Kit/s 0.0s
/tmp/coco128/images/train2017/000000000009.jpg: ignoring corrupt image/label: Label class 50 exceeds dataset class count 4. Possible class labels are 0-3
/tmp/coco128/images/train2017/000000000025.jpg: ignoring corrupt image/label: Label class 23 exceeds dataset class count 4. Possible class labels are 0-3
/tmp/coco128/images/train2017/000000000030.jpg: ignoring corrupt image/label: Label class 75 exceeds dataset class count 4. Possible class labels are 0-3
/tmp/coco128/images/train2017/000000000034.jpg: ignoring corrupt image/label: Label class 22 exceeds dataset clas

2026-03-29 13:19:30.387866533 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2837629, index: 0, mask: {1, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-29 13:19:30.387939603 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2837630, index: 1, mask: {2, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-29 13:19:30.387948695 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2837631, index: 2, mask: {3, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-29 13:19:30.387956476 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2837632, index: 3, mask: {4, }, error code: 22 error msg: Invalid argument. Specify the n

Scanning /tmp/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 123 corrupt: 100% ━━━━━━━━━━━━ 128/128 28.3Mit/s 0.0s
/tmp/coco128/images/train2017/000000000009.jpg: ignoring corrupt image/label: Label class 50 exceeds dataset class count 4. Possible class labels are 0-3
/tmp/coco128/images/train2017/000000000025.jpg: ignoring corrupt image/label: Label class 23 exceeds dataset class count 4. Possible class labels are 0-3
/tmp/coco128/images/train2017/000000000030.jpg: ignoring corrupt image/label: Label class 75 exceeds dataset class count 4. Possible class labels are 0-3
/tmp/coco128/images/train2017/000000000034.jpg: ignoring corrupt image/label: Label class 22 exceeds dataset class count 4. Possible class labels are 0-3
/tmp/coco128/images/train2017/000000000036.jpg: ignoring corrupt image/label: Label class 25 exceeds dataset class count 4. Possible class labels are 0-3
/tmp/coco128/images/train2017/000000000042.jpg: ignoring corrupt image/label: Label class 16 exceeds 

In [15]:
import time
import torch
import numpy as np
from ultralytics import YOLO

def benchmark(model_path, label, runs=200, warmup=20):
    model = YOLO(model_path)
    dummy = "/tmp/coco128/images/train2017/000000000001.jpg"
    # Use a real image path that exists
    import glob
    imgs = glob.glob("/tmp/coco128/images/train2017/*.jpg")
    img = imgs[0] if imgs else "https://ultralytics.com/images/bus.jpg"

    # Warmup
    for _ in range(warmup):
        model(img, verbose=False)

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs):
        model(img, verbose=False)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) / runs * 1000
    return elapsed

print("Benchmarking... (each model runs 200 inferences)\n")

results = {}

print("1/6 YOLO11 PyTorch FP32...")
results["YOLO11 PyTorch"] = benchmark("/home/jj7/Scalable/yolo11s_pruned.pt",    "YOLO11 PyTorch")

print("2/6 YOLO11 FP16 engine...")
results["YOLO11 FP16"]   = benchmark("/home/jj7/Scalable/yolo11s_fp16.engine",  "YOLO11 FP16")

print("3/6 YOLO11 INT8 engine...")
results["YOLO11 INT8"]   = benchmark("/home/jj7/Scalable/yolo11s_pruned.engine","YOLO11 INT8")

print("4/6 YOLO26 PyTorch FP32...")
results["YOLO26 PyTorch"] = benchmark("/home/jj7/Scalable/YOLO26_pruned_20_final.pt",     "YOLO26 PyTorch")

print("5/6 YOLO26 FP16 engine...")
results["YOLO26 FP16"]   = benchmark("/home/jj7/Scalable/YOLO26_fp16.engine",   "YOLO26 FP16")

print("6/6 YOLO26 INT8 engine...")
results["YOLO26 INT8"]   = benchmark("/home/jj7/Scalable/YOLO26_pruned_20_final.engine","YOLO26 INT8")

# ── Print results table ───────────────────────────────────────────
print("\n" + "="*60)
print(f"{'Model':<22} {'Latency':>10} {'FPS':>8} {'Speedup':>10}")
print("="*60)

base11 = results["YOLO11 PyTorch"]
base26 = results["YOLO26 PyTorch"]

for name, ms in results.items():
    base = base11 if "11" in name else base26
    fps  = 1000 / ms
    spd  = base / ms
    print(f"{name:<22} {ms:>8.2f}ms {fps:>8.1f} {spd:>9.2f}x")

print("="*60)
print(f"\n✅ Best YOLO11: INT8 at {results['YOLO11 INT8']:.2f}ms")
print(f"✅ Best YOLO26: INT8 at {results['YOLO26 INT8']:.2f}ms")

Benchmarking... (each model runs 200 inferences)

1/6 YOLO11 PyTorch FP32...
2/6 YOLO11 FP16 engine...
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading /home/jj7/Scalable/yolo11s_fp16.engine for TensorRT inference...
[03/29/2026-13:21:01] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
[03/29/2026-13:21:01] [TRT] [I] Loaded engine size: 18 MiB
[03/29/2026-13:21:01] [TRT] [I] [MS] Running engine with multi stream info
[03/29/2026-13:21:01] [TRT] [I] [MS] Number of aux streams is 4
[03/29/2026-13:21:01] [TRT] [I] [MS] Number of total worker streams is 5
[03/29/2026-13:21:01] [TRT] [I] [MS] The main stream provided by execute/enqueue cal

In [17]:
from ultralytics import YOLO

# Load best engines
engine11 = YOLO("/home/jj7/Scalable/yolo11s_fp16.engine", task="detect")
engine26 = YOLO("/home/jj7/Scalable/YOLO26_fp16.engine",  task="detect")

# Test on a sample image
import glob
test_img = glob.glob("/tmp/coco128/images/train2017/*.jpg")[0]

print("=== YOLO11s FP16 prediction ===")
r11 = engine11(test_img, verbose=True)

print("\n=== YOLO26 FP16 prediction ===")
r26 = engine26(test_img, verbose=True)

print("\n=== Files ready for deployment ===")
import os
for f in sorted(os.listdir("/home/jj7/Scalable")):
    if f.endswith(".engine"):
        size = os.path.getsize(f"/home/jj7/Scalable/{f}") / 1e6
        print(f"  {f:45s}  {size:.1f} MB")

=== YOLO11s FP16 prediction ===
Loading /home/jj7/Scalable/yolo11s_fp16.engine for TensorRT inference...
[03/29/2026-13:25:00] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
[03/29/2026-13:25:00] [TRT] [I] Loaded engine size: 18 MiB
[03/29/2026-13:25:00] [TRT] [I] [MS] Running engine with multi stream info
[03/29/2026-13:25:00] [TRT] [I] [MS] Number of aux streams is 4
[03/29/2026-13:25:00] [TRT] [I] [MS] Number of total worker streams is 5
[03/29/2026-13:25:00] [TRT] [I] [MS] The main stream provided by execute/enqueue calls is the first worker stream
[03/29/2026-13:25:00] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +27, now: CPU 0, GPU 134 (MiB)
WARNING ⚠️ Metadata not found for 'model=/home/jj7/Scalable/yolo1

In [19]:
import zipfile
import os

# Create zip with all quantized engine files
zip_path = "/home/jj7/Scalable/quantized_engines.zip"

files_to_zip = [
    "/home/jj7/Scalable/yolo11s_pruned.engine",
    "/home/jj7/Scalable/yolo11s_fp16.engine",
    "/home/jj7/Scalable/YOLO26_pruned_20_final.engine",
    "/home/jj7/Scalable/YOLO26_fp16.engine",
]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in files_to_zip:
        fname = os.path.basename(fpath)
        zf.write(fpath, fname)
        print(f"  Added: {fname}  ({os.path.getsize(fpath)/1e6:.1f} MB)")

total = os.path.getsize(zip_path) / 1e6
print(f"\n✅ ZIP created: {zip_path}")
print(f"   Total size: {total:.1f} MB")

# Auto-trigger browser download
from IPython.display import Javascript, display
display(Javascript(f"""
var a = document.createElement('a');
a.href = '/files/Scalable/quantized_engines.zip';
a.download = 'quantized_engines.zip';
document.body.appendChild(a);
a.click();
document.body.removeChild(a);
"""))
print("⬇️  Download should start automatically in your browser!")

  Added: yolo11s_pruned.engine  (16.3 MB)
  Added: yolo11s_fp16.engine  (19.7 MB)
  Added: YOLO26_pruned_20_final.engine  (18.0 MB)
  Added: YOLO26_fp16.engine  (20.5 MB)

✅ ZIP created: /home/jj7/Scalable/quantized_engines.zip
   Total size: 47.7 MB


<IPython.core.display.Javascript object>

⬇️  Download should start automatically in your browser!
